# Query Selection And Trace-Based Error Analysis

This notebook connects final-output evaluation with LangSmith/LangGraph trace behavior. The goal is to pick a small set of queries that make a strong error analysis instead of cherry-picking only the best or worst examples.

The analysis uses three layers:

1. Final plan quality from `data/evaluation/{baseline,travel_agent}` scorecards.
2. Final artifacts from `data/travelplans/{baseline,travel_agent}`.
3. Trace/process indicators from `data/traces`, especially Travel Agent real `run_type == tool` rows.

Important caveat from `thread_analysis_context.txt`: the full Travel Agent trace export contains many duplicated message/tool snapshots. For execution-level tool analysis, this notebook uses `langsmith_runs.csv` filtered to `run_type == "tool"`, rather than counting every row in the recursive `tool_calls.csv` export.

In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)


## 1. Paths And Taxonomy

These constants mirror the existing `langsmith_traces_analysis` code, but point at this repository's canonical `data/traces` directory. Tool groups follow the existing paper/error-analysis taxonomy: domain search, TravelPlan mutation, and state management.

In [ ]:
ROOT = Path("..").resolve()
DATA = ROOT / "data"
EVAL_DIR = DATA / "evaluation"
PLAN_DIR = DATA / "travelplans"
TRACE_DIR = DATA / "traces"

TP_EVAL_PATH = EVAL_DIR / "travel_agent"
BL_EVAL_PATH = EVAL_DIR / "baseline"
TP_PLAN_PATH = PLAN_DIR / "travel_agent"
BL_PLAN_PATH = PLAN_DIR / "baseline"
TP_TRACE_PATH = TRACE_DIR / "travel_agent_full"
BL_TRACE_PATH = TRACE_DIR / "baseline"
QUERY_CATALOG_PATH = DATA / "travel_queries_all_30.json"

SEARCH_TOOLS = {
    "search_web",
    "search_restaurants",
    "search_attractions",
    "search_flights",
    "search_hotels",
    "check_route_timing",
    "build_place_distance_graph",
    "tavily_web_search",
}
MUTATION_TOOLS = {"init_plan", "add_day", "remove_day", "add_slot", "insert_slot", "delete_slot", "view_plan", "cost_summary"}
STATE_TOOLS = {"write_todos"}
VALID_CATEGORIES = {"meal", "attraction", "transport", "lodging", "leisure", "other"}

# These regexes are copied from the prior trace-analysis scripts and are intentionally broad.
# They are indicators, not proof: use the drilldown cells below before making claims in the paper.
ERROR_RE = re.compile(r"timeout|timed out|error|failed|failure|exception|traceback|rate limit|unavailable|could not|unable|no reliable|missing info|missing_info", re.I)
UNCERTAIN_RE = re.compile(r"verify|re-verify|estimated|estimate|uncertain|unavailable|not available|backup|fallback|could not|depends|must be checked|not confirmed", re.I)
FRAGILE_DOMAINS_RE = re.compile(r"google\.com|maps\.google|instagram\.com|facebook\.com|tripadvisor\.com|booking\.com|expedia\.com|agoda\.com|turbopass\.com", re.I)


## 2. Reusable Normalization Helpers

These helpers are adapted from `langsmith_traces_analysis/scripts/make_correct_error_analysis_notebook.py` and `make_latex_error_tables.py`. They normalize JSON arguments, query strings, durations, and tool groups so baseline and Travel Agent traces can be compared consistently.

In [ ]:
def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def safe_loads(value: Any) -> Any:
    """Parse JSON-ish values from CSV cells without failing the notebook."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return {}
    if isinstance(value, (dict, list)):
        return value
    try:
        return json.loads(value)
    except Exception:
        return {"_raw": str(value)}


def canonical(obj: Any) -> str:
    """Canonical text for repeat detection. URLs and punctuation are normalized away."""
    if obj is None or (isinstance(obj, float) and np.isnan(obj)):
        return ""
    if not isinstance(obj, str):
        obj = json.dumps(obj, ensure_ascii=False, sort_keys=True, default=str)
    obj = obj.lower()
    obj = re.sub(r"https?://\S+", " ", obj)
    obj = re.sub(r"[^\w\s€$£.-]", " ", obj)
    return re.sub(r"\s+", " ", obj).strip()


def arg_text(args: Any) -> str:
    """Extract the semantically important query/request part of a tool call."""
    if not isinstance(args, dict):
        return canonical(args)
    for key in ["query", "request", "text", "prompt", "origin", "destination", "place", "location"]:
        if args.get(key):
            return canonical(args[key])
    return canonical(args)


def duration_seconds(start: Any, end: Any) -> float:
    try:
        return (pd.to_datetime(end, utc=True) - pd.to_datetime(start, utc=True)).total_seconds()
    except Exception:
        return np.nan


def tool_group(name: str) -> str:
    if name in SEARCH_TOOLS:
        return "Domain/search tools"
    if name in MUTATION_TOOLS:
        return "TravelPlan mutation tools"
    if name in STATE_TOOLS:
        return "State-management tools"
    return "Other"


def short_query_id(full_id: str) -> str:
    # query_1_couple_citytrip_Adrian -> query_1
    return "_".join(full_id.split("_")[:2])


def pass_rate(pass_count: int, total: int) -> float | None:
    return round(pass_count / total, 3) if total else None


## 3. Load Final Evaluations And Outputs

This builds a query-level table with scorecard quality metrics and final artifacts. We keep both loaded final outputs in `query_dict` so the drilldown cells can show the baseline markdown and structured Travel Agent JSON for any selected query.

In [ ]:
def scorecard_summary(scorecard: dict[str, Any]) -> dict[str, Any]:
    checks = scorecard.get("aggregated_constraints") or []
    verdicts = Counter(check.get("final_verdict", "UNKNOWN") for check in checks)
    hard_checks = [check for check in checks if check.get("constraint_type") == "hard"]
    hard_verdicts = Counter(check.get("final_verdict", "UNKNOWN") for check in hard_checks)
    rationale_total = sum(scorecard.get(key, 0) for key in ["rationale_pass_count", "rationale_fail_count", "rationale_missing_count"])
    total = sum(verdicts.values())
    hard_total = sum(hard_verdicts.values())
    return {
        "scorecard_checks": total,
        "scorecard_pass": verdicts.get("PASS", 0),
        "scorecard_fail": verdicts.get("FAIL", 0),
        "scorecard_missing": verdicts.get("MISSING_INFO", 0),
        "scorecard_pass_rate": pass_rate(verdicts.get("PASS", 0), total),
        "hard_checks": hard_total,
        "hard_pass": hard_verdicts.get("PASS", 0),
        "hard_fail": hard_verdicts.get("FAIL", 0),
        "hard_missing": hard_verdicts.get("MISSING_INFO", 0),
        "hard_pass_rate": pass_rate(hard_verdicts.get("PASS", 0), hard_total),
        "hc_micro_pass_rate": scorecard.get("hc_micro_pass_rate"),
        "hc_macro_pass_rate": scorecard.get("hc_macro_pass_rate"),
        "rationale_checks": rationale_total,
        "rationale_pass": scorecard.get("rationale_pass_count", 0),
        "rationale_fail": scorecard.get("rationale_fail_count", 0),
        "rationale_missing": scorecard.get("rationale_missing_count", 0),
        "rationale_pass_rate": pass_rate(scorecard.get("rationale_pass_count", 0), rationale_total),
    }


def load_final_outputs() -> tuple[dict[str, dict[str, Any]], pd.DataFrame]:
    queries = load_json(QUERY_CATALOG_PATH)
    query_by_short_id = {short_query_id(q["id"]): q for q in queries}
    query_dict: dict[str, dict[str, Any]] = {}
    rows = []

    for tp_eval_file in sorted(TP_EVAL_PATH.glob("*.json"), key=lambda p: int(short_query_id(p.stem).split("_")[1])):
        query_id = short_query_id(tp_eval_file.stem)
        tp_eval = load_json(tp_eval_file)
        bl_eval = load_json(BL_EVAL_PATH / tp_eval_file.name)
        tp_scorecard = load_json(TP_EVAL_PATH / tp_eval_file.stem / "scorecard.json")
        bl_scorecard = load_json(BL_EVAL_PATH / tp_eval_file.stem / "scorecard.json")
        tp_plan_path = ROOT / tp_eval["plan_path"]
        bl_plan_path = ROOT / bl_eval["plan_path"]
        tp_output = load_json(tp_plan_path)
        bl_output = bl_plan_path.read_text(encoding="utf-8")

        query_dict[query_id] = {
            "query": query_by_short_id[query_id]["query"],
            "type": query_by_short_id[query_id]["type"],
            "hard_constraints": query_by_short_id[query_id]["hard_constraints"],
            "tp_eval": tp_eval,
            "bl_eval": bl_eval,
            "tp_scorecard": tp_scorecard,
            "bl_scorecard": bl_scorecard,
            "tp_plan_path": tp_plan_path,
            "bl_plan_path": bl_plan_path,
            "tp_output": tp_output,
            "bl_output": bl_output,
        }

        for system, eval_obj, scorecard in [("baseline", bl_eval, bl_scorecard), ("travel_agent", tp_eval, tp_scorecard)]:
            rows.append({
                "query_id": query_id,
                "system": system,
                "query_type": query_by_short_id[query_id]["type"],
                "status": eval_obj.get("status"),
                "duration_seconds": eval_obj.get("duration_seconds"),
                "has_timeout": eval_obj.get("has_timeout"),
                **scorecard_summary(scorecard),
            })

    return query_dict, pd.DataFrame(rows)


query_dict, final_eval = load_final_outputs()
display(final_eval.head())
display(Markdown(f"Loaded **{len(query_dict)} queries** with baseline and Travel Agent final outputs."))


## 4. Load And Normalize Trace Tables

For Travel Agent, use `langsmith_runs.csv` filtered to real tool runs. For baseline, use `local_tool_calls.csv` request rows from the local message export; baseline does not have the same rich execution-agent trace, so comparisons should be framed as final-answer plus visible Tavily-call behavior.

In [ ]:
# Match traces to evaluated queries by the user query string.
# Do not use root_run_id here: baseline and Travel Agent have different run ids for the same query.
query_lookup = pd.DataFrame([
    {"query_id": query_id, "query": info["query"], "query_norm_for_diagnostics": canonical(info["query"])}
    for query_id, info in query_dict.items()
])

tp_manifest = pd.read_csv(TP_TRACE_PATH / "manifest.csv")
tp_runs_all = pd.read_csv(TP_TRACE_PATH / "langsmith_runs.csv")
bl_manifest = pd.read_csv(BL_TRACE_PATH / "manifest.csv")
bl_tools_raw = pd.read_csv(BL_TRACE_PATH / "local_tool_calls.csv")

# Travel Agent: real tool executions only. This avoids overcounting recursive snapshots in the full export.
tp_tools = tp_runs_all[tp_runs_all["run_type"] == "tool"].copy()
tp_tools["system"] = "travel_agent"
tp_tools["tool_args"] = tp_tools["inputs_json"].map(safe_loads)
tp_tools["args_norm"] = tp_tools["tool_args"].map(canonical)
tp_tools["query_norm"] = tp_tools["tool_args"].map(arg_text)
tp_tools["output_text"] = tp_tools["outputs_json"].fillna("").astype(str)
tp_tools["duration_s"] = [duration_seconds(s, e) for s, e in zip(tp_tools["start_time"], tp_tools["end_time"])]
tp_tools["has_error_language"] = tp_tools["output_text"].str.contains(ERROR_RE, na=False)
tp_tools["has_uncertainty_language"] = tp_tools["output_text"].str.contains(UNCERTAIN_RE, na=False)
tp_tools = tp_tools.merge(tp_manifest[["thread_id", "root_run_id", "query", "validation_passed", "validation_attempts", "travelplan_title"]], on="thread_id", how="left")
tp_tools = tp_tools.merge(query_lookup[["query_id", "query"]], on="query", how="left")
tp_tools["agent_group"] = tp_tools["name"].map(tool_group)
tp_tools = tp_tools.sort_values(["thread_id", "start_time", "id"]).reset_index(drop=True)
tp_tools["call_index"] = tp_tools.groupby("thread_id").cumcount()

# Baseline: keep requested Tavily calls and join the corresponding query metadata.
bl_tools = bl_tools_raw[bl_tools_raw["source"] == "ai_tool_call_request"].copy()
bl_tools["system"] = "baseline"
bl_tools["name"] = bl_tools["tool_name"]
bl_tools["tool_args"] = bl_tools["tool_args_json"].map(safe_loads)
bl_tools["args_norm"] = bl_tools["tool_args"].map(canonical)
bl_tools["query_norm"] = bl_tools["tool_args"].map(arg_text)
bl_tools["duration_s"] = np.nan
bl_tools["output_text"] = ""
bl_tools["has_error_language"] = False
bl_tools["has_uncertainty_language"] = False
bl_tools = bl_tools.merge(bl_manifest[["thread_id", "root_run_id", "query", "requested_tool_calls", "executed_tool_messages", "final_markdown_chars"]], on="thread_id", how="left")
bl_tools = bl_tools.merge(query_lookup[["query_id", "query"]], on="query", how="left")
bl_tools["agent_group"] = bl_tools["name"].map(tool_group)

display(Markdown(f"Travel Agent real tool runs: **{len(tp_tools):,}** across **{tp_tools.thread_id.nunique()}** threads."))
display(Markdown(f"Baseline requested tool calls: **{len(bl_tools):,}** across **{bl_tools.thread_id.nunique()}** threads."))
display(Markdown("### Query-string match diagnostics"))
display(pd.DataFrame([
    {"system": "travel_agent", "trace_queries": tp_manifest["query"].nunique(), "matched_queries": tp_tools["query_id"].nunique(), "unmatched_tool_rows": int(tp_tools["query_id"].isna().sum())},
    {"system": "baseline", "trace_queries": bl_manifest["query"].nunique(), "matched_queries": bl_tools["query_id"].nunique(), "unmatched_tool_rows": int(bl_tools["query_id"].isna().sum())},
]))
display(tp_tools[["query_id", "thread_id", "name", "agent_group", "duration_s", "query_norm", "has_error_language", "has_uncertainty_language"]].head())


## 5. Error Indicator Functions

These functions compute the same operational indicators described in `thread_analysis_context.txt`: repeated calls, repeated search queries, loop/backtracking indicators, timeout/error-language indicators, uncertainty outputs, and cascade indicators. They intentionally produce counts to rank candidates; the selected examples still need manual drilldown before writing causal claims.

In [ ]:
def repeated_exact_calls(tools: pd.DataFrame) -> pd.DataFrame:
    if tools.empty:
        return pd.DataFrame(columns=["thread_id", "name", "args_norm", "count"])
    return (
        tools.groupby(["thread_id", "name", "args_norm"], dropna=False)
        .agg(count=("name", "size"), query_id=("query_id", "first"), first_call=("call_index", "min"), last_call=("call_index", "max"))
        .reset_index()
        .query("args_norm != '' and count > 1")
        .sort_values("count", ascending=False)
    )


def repeated_search_queries(tools: pd.DataFrame) -> pd.DataFrame:
    search = tools[tools["name"].isin(SEARCH_TOOLS)].copy()
    if search.empty:
        return pd.DataFrame(columns=["thread_id", "name", "query_norm", "count"])
    return (
        search.groupby(["thread_id", "name", "query_norm"], dropna=False)
        .agg(count=("name", "size"), query_id=("query_id", "first"), first_call=("call_index", "min"), last_call=("call_index", "max"))
        .reset_index()
        .query("query_norm != '' and count > 1")
        .sort_values("count", ascending=False)
    )


def loop_indicators(tools: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for thread_id, group in tools.groupby("thread_id"):
        ordered = group.sort_values("call_index").to_dict("records")
        streak = 1
        prev_key = None
        for row in ordered:
            key = (row.get("name"), row.get("args_norm") or row.get("query_norm"))
            streak = streak + 1 if key == prev_key else 1
            if streak >= 2:
                rows.append({"thread_id": thread_id, "query_id": row.get("query_id"), "name": row.get("name"), "loop_type": "consecutive exact repeat", "call_index": row.get("call_index")})
            prev_key = key

        for i in range(len(ordered) - 3):
            names = [r.get("name") for r in ordered[i:i + 4]]
            if names in (["add_slot", "delete_slot", "add_slot", "delete_slot"], ["insert_slot", "delete_slot", "insert_slot", "delete_slot"], ["delete_slot", "add_slot", "delete_slot", "add_slot"], ["delete_slot", "insert_slot", "delete_slot", "insert_slot"]):
                rows.append({"thread_id": thread_id, "query_id": ordered[i].get("query_id"), "name": "mutation repair", "loop_type": "add/delete backtrack", "call_index": ordered[i].get("call_index")})

        counts = Counter((r.get("name"), r.get("query_norm") or r.get("args_norm")) for r in ordered)
        for (name, key_text), count in counts.items():
            if key_text and count >= 3:
                qid = group["query_id"].dropna().iloc[0] if group["query_id"].notna().any() else None
                rows.append({"thread_id": thread_id, "query_id": qid, "name": name, "loop_type": "same call >=3", "call_index": np.nan})

    return pd.DataFrame(rows)


def cascade_indicators(tools: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for thread_id, group in tools.groupby("thread_id"):
        group = group.sort_values("call_index")
        meta = group.iloc[0]
        if meta.get("validation_attempts", 1) > 1:
            rows.append({"thread_id": thread_id, "query_id": meta.get("query_id"), "cascade": "validator repair loop"})

        weak_calls = group[group["has_uncertainty_language"] | group["has_error_language"]]["call_index"]
        for idx in weak_calls:
            later_mutation = group[(group["call_index"] > idx) & (group["name"].isin({"add_slot", "insert_slot"}))]
            if not later_mutation.empty:
                rows.append({"thread_id": thread_id, "query_id": meta.get("query_id"), "cascade": "weak evidence then plan mutation"})

        for _, row in group[group["name"] == "delete_slot"].iterrows():
            prior_insert = group[(group["call_index"] < row["call_index"]) & (group["name"].isin({"add_slot", "insert_slot"}))]
            if not prior_insert.empty:
                rows.append({"thread_id": thread_id, "query_id": meta.get("query_id"), "cascade": "delete after prior insertion"})

    return pd.DataFrame(rows)


tp_repeats = repeated_exact_calls(tp_tools)
tp_query_repeats = repeated_search_queries(tp_tools)
tp_loops = loop_indicators(tp_tools)
tp_cascades = cascade_indicators(tp_tools)

bl_repeats = repeated_exact_calls(bl_tools)
bl_query_repeats = repeated_search_queries(bl_tools)

display(Markdown(f"Travel Agent repeated exact call groups: **{len(tp_repeats)}**; loop indicators: **{len(tp_loops)}**; cascade indicators: **{len(tp_cascades)}**."))
display(tp_repeats.head(10))


## 6. Query-Level Trace Summary

This table is the main bridge between final quality and process behavior. It makes each query selectable based on both final errors and execution symptoms.

In [ ]:
def count_by_query(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    if df.empty or "query_id" not in df.columns:
        return pd.DataFrame(columns=["query_id", value_name])
    return df.groupby("query_id").size().reset_index(name=value_name)


tp_trace_summary = (
    tp_tools.groupby("query_id")
    .agg(
        tp_tool_calls=("name", "size"),
        tp_search_calls=("agent_group", lambda s: (s == "Domain/search tools").sum()),
        tp_mutation_calls=("agent_group", lambda s: (s == "TravelPlan mutation tools").sum()),
        tp_state_calls=("agent_group", lambda s: (s == "State-management tools").sum()),
        tp_error_outputs=("has_error_language", "sum"),
        tp_uncertainty_outputs=("has_uncertainty_language", "sum"),
        tp_p95_latency_s=("duration_s", lambda s: round(s.quantile(0.95), 2)),
        validation_attempts=("validation_attempts", "max"),
        validation_passed=("validation_passed", "first"),
    )
    .reset_index()
)

bl_trace_summary = (
    bl_tools.groupby("query_id")
    .agg(
        bl_tool_calls=("name", "size"),
        bl_search_calls=("agent_group", lambda s: (s == "Domain/search tools").sum()),
        bl_final_markdown_chars=("final_markdown_chars", "max"),
    )
    .reset_index()
)

trace_summary = tp_trace_summary.merge(bl_trace_summary, on="query_id", how="outer")
for df, col in [(tp_repeats, "tp_repeated_exact_groups"), (tp_query_repeats, "tp_repeated_query_groups"), (tp_loops, "tp_loop_indicators"), (tp_cascades, "tp_cascade_indicators"), (bl_repeats, "bl_repeated_exact_groups"), (bl_query_repeats, "bl_repeated_query_groups")]:
    trace_summary = trace_summary.merge(count_by_query(df, col), on="query_id", how="left")

trace_summary = trace_summary.fillna(0)

eval_wide = final_eval.pivot(index="query_id", columns="system")
eval_wide.columns = [f"{system}_{metric}" for metric, system in eval_wide.columns]
eval_wide = eval_wide.reset_index()

# Start from all evaluated queries, then left-join trace indicators.
# This keeps queries in the table even if trace data is absent or failed to match.
query_selection_table = eval_wide.merge(trace_summary, on="query_id", how="left")
query_selection_table = query_selection_table.fillna(0)

# A pragmatic ranking score: high means either final quality is weak, trace behavior is unstable, or the two systems contrast sharply.
query_selection_table["final_error_score"] = (
    query_selection_table["travel_agent_scorecard_fail"].fillna(0) * 3
    + query_selection_table["travel_agent_hard_fail"].fillna(0) * 5
    + query_selection_table["travel_agent_rationale_fail"].fillna(0) * 2
    + query_selection_table["travel_agent_rationale_missing"].fillna(0)
)
query_selection_table["trace_error_score"] = (
    query_selection_table["tp_loop_indicators"].fillna(0) * 2
    + query_selection_table["tp_repeated_exact_groups"].fillna(0)
    + query_selection_table["tp_error_outputs"].fillna(0)
    + query_selection_table["tp_uncertainty_outputs"].fillna(0)
    + (query_selection_table["validation_attempts"].fillna(1).clip(lower=1) - 1) * 5
)
query_selection_table["baseline_vs_tp_contrast"] = (
    query_selection_table["baseline_scorecard_fail"].fillna(0)
    - query_selection_table["travel_agent_scorecard_fail"].fillna(0)
    + query_selection_table["baseline_hard_fail"].fillna(0)
    - query_selection_table["travel_agent_hard_fail"].fillna(0)
)
query_selection_table["selection_score"] = (
    query_selection_table["final_error_score"]
    + 0.25 * query_selection_table["trace_error_score"]
    + query_selection_table["baseline_vs_tp_contrast"].abs()
)

cols = [
    "query_id", "baseline_scorecard_pass_rate", "travel_agent_scorecard_pass_rate",
    "baseline_hard_pass_rate", "travel_agent_hard_pass_rate",
    "baseline_rationale_pass_rate", "travel_agent_rationale_pass_rate",
    "tp_tool_calls", "tp_mutation_calls", "tp_loop_indicators", "tp_cascade_indicators",
    "tp_error_outputs", "tp_uncertainty_outputs", "validation_attempts",
    "bl_tool_calls", "selection_score",
]
display(query_selection_table[cols].sort_values("selection_score", ascending=False))


## 7. Recommended Query Set

A strong error analysis should include contrast, not just worst cases. This cell proposes a compact set of case studies:

- **Normal anchor**: `query_1`, because it is easy to explain and already used in the structure comparison.
- **Highest trace instability**: query with the largest trace error score.
- **Largest baseline-vs-Travel-Agent contrast**: query where one system clearly outperforms the other on scorecard/hard constraints.
- **Worst Travel Agent rationale grounding**: query with low rationale pass rate or many missing rationale checks.
- **Domain/edge-case query**: highest selected candidate among non-normal query types, useful for showing generalization issues.

In [ ]:
def first_query_id(df: pd.DataFrame, sort_col: str, ascending: bool = False) -> str | None:
    if df.empty:
        return None
    return df.sort_values(sort_col, ascending=ascending).iloc[0]["query_id"]

recommendations = []
if "query_1" in set(query_selection_table["query_id"]):
    recommendations.append({"role": "normal anchor / readable baseline", "query_id": "query_1"})
recommendations.append({"role": "highest trace instability", "query_id": first_query_id(query_selection_table, "trace_error_score")})
recommendations.append({"role": "largest baseline vs Travel Agent contrast", "query_id": first_query_id(query_selection_table.assign(abs_contrast=query_selection_table["baseline_vs_tp_contrast"].abs()), "abs_contrast")})
recommendations.append({"role": "weakest Travel Agent rationale grounding", "query_id": first_query_id(query_selection_table, "travel_agent_rationale_pass_rate", ascending=True)})
non_normal = query_selection_table[query_selection_table["query_id"].map(lambda qid: query_dict[qid]["type"] != "normal")]
recommendations.append({"role": "edge/domain-specific case", "query_id": first_query_id(non_normal, "selection_score")})

recommended_queries = pd.DataFrame(recommendations).dropna().drop_duplicates(["role", "query_id"])
recommended_queries["query_type"] = recommended_queries["query_id"].map(lambda qid: query_dict[qid]["type"])
recommended_queries["query"] = recommended_queries["query_id"].map(lambda qid: query_dict[qid]["query"])
display(recommended_queries)

display(Markdown("### Suggested interpretation"))
display(Markdown(
    "Use `query_1` as the narrative anchor, then add 2-3 high-scoring cases from this table. "
    "For each selected query, report final scorecard/rationale outcomes first, then use the trace timeline to explain likely process-level causes: weak evidence, repeated mutation, validator repair, or search/tool uncertainty."
))


## 8. Single-Query Drilldown

Change `SELECTED_QUERY_ID` to any recommended query. The cells below show the final scores, Travel Agent tool timeline, error-indicator rows, and final outputs. Use this section to write the actual case-study paragraphs.

In [ ]:
SELECTED_QUERY_ID = "query_1"

selected_eval = final_eval[final_eval["query_id"] == SELECTED_QUERY_ID].copy()
selected_trace = query_selection_table[query_selection_table["query_id"] == SELECTED_QUERY_ID].copy()
display(Markdown(f"## Drilldown: `{SELECTED_QUERY_ID}`"))
display(Markdown(query_dict[SELECTED_QUERY_ID]["query"]))
display(selected_eval[["system", "scorecard_pass_rate", "hard_pass_rate", "rationale_pass_rate", "rationale_pass", "rationale_fail", "rationale_missing"]])
display(selected_trace[["tp_tool_calls", "tp_search_calls", "tp_mutation_calls", "tp_loop_indicators", "tp_cascade_indicators", "tp_error_outputs", "tp_uncertainty_outputs", "validation_attempts", "bl_tool_calls"]])


In [ ]:
# Timeline view: this is the fastest way to inspect what happened before the final TravelPlan.
timeline_cols = ["call_index", "name", "agent_group", "duration_s", "query_norm", "has_error_language", "has_uncertainty_language"]
selected_tp_timeline = tp_tools[tp_tools["query_id"] == SELECTED_QUERY_ID].sort_values("call_index")
display(selected_tp_timeline[timeline_cols].head(80))

display(Markdown("### Selected query error indicators"))
display(Markdown("Repeated exact Travel Agent calls"))
display(tp_repeats[tp_repeats["query_id"] == SELECTED_QUERY_ID].head(30))
display(Markdown("Loop indicators"))
display(tp_loops[tp_loops["query_id"] == SELECTED_QUERY_ID].head(30))
display(Markdown("Cascade indicators"))
display(tp_cascades[tp_cascades["query_id"] == SELECTED_QUERY_ID].head(30))


## 9. Final Artifact Side-By-Side

Use this for qualitative write-up after the quantitative table. Keep the baseline final answer and the Travel Agent final structured artifact visible while inspecting trace indicators above.

In [ ]:
display(Markdown("### Baseline final answer"))
display(Markdown(query_dict[SELECTED_QUERY_ID]["bl_output"][:6000]))

display(Markdown("### Travel Agent final TravelPlan summary"))
tp_output = query_dict[SELECTED_QUERY_ID]["tp_output"]
travelplan = tp_output.get("travelplan", {})
slot_rows = []
for day in travelplan.get("days", []):
    for slot in day.get("slots", []):
        slot_rows.append({
            "day": day.get("index"),
            "name": slot.get("name"),
            "category": slot.get("category"),
            "start_time": slot.get("start_time"),
            "end_time": slot.get("end_time"),
            "cost": slot.get("cost"),
            "links": len(slot.get("links") or []),
            "notes": slot.get("notes"),
        })
display(pd.DataFrame(slot_rows))
display(Markdown(f"Validation passed: **{tp_output.get('validation_passed')}**; attempts: **{tp_output.get('validation_attempts')}**"))
display(Markdown(str(tp_output.get("validation_feedback", ""))))


## 10. How To Turn This Into The Error Analysis

For each chosen query, write the case study in this order:

1. **Final artifact outcome**: hard constraints, rationale pass/missing/fail, complete answer status, and Travel Agent validation status.
2. **Trace symptoms**: number of tool calls, mutation calls, repeated calls, loop indicators, weak evidence, and validation attempts.
3. **Causal hypothesis**: connect trace symptoms to final errors, but phrase as evidence-backed likelihood unless manually inspected.
4. **Architecture implication**: baseline is simpler and less observable; Travel Agent has stronger structure and validation but can propagate weak evidence through repeated TravelPlan mutations.

The strongest paper claim to test with this notebook is: Travel Agent improves structural validity and hard-constraint satisfaction, but its main failure mode shifts from missing final-answer structure to weak-evidence propagation and state-repair/mutation loops.